# 08f: Visual Invariance Notebook

## Purpose
Provide visual proof that DSL produces bit-identical results to direct ROOT operations.

## Phase 13.6.G Debug Notebook

This notebook uses A/B/A-B comparison pattern:
- **A**: DSL result
- **B**: ROOT direct / NumPy baseline
- **A-B**: Difference (should be ~0)

## Setup

In [ ]:
# Path setup - ensure RDataFrameDSL is importable
import sys
import os

# Add parent directory to path if running from examples/
notebook_dir = os.path.dirname(os.path.abspath('.'))
if 'RDataFrameDSL' not in sys.modules:
    for path in ['.', '..', notebook_dir]:
        if os.path.exists(os.path.join(path, 'RDataFrameDSL')):
            sys.path.insert(0, os.path.abspath(path))
            break

In [ ]:
import ROOT
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging

from RDataFrameDSL import DSLCompiler

logging.basicConfig(level=logging.INFO)
plt.rcParams['figure.figsize'] = (12, 3)

print("Setup complete")

In [ ]:
def visual_invariance_check(name, dsl_result, root_result, tolerance=1e-10):
    """
    Visual A/B/A-B comparison for invariance verification.
    
    Args:
        name: Operation name
        dsl_result: Array from DSL
        root_result: Array from direct ROOT/NumPy
        tolerance: Numerical tolerance for comparison
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # A: DSL Result
    axes[0].hist(dsl_result, bins=50, alpha=0.7, color='blue', label='DSL')
    axes[0].set_title(f"A: DSL Result\n{name}")
    axes[0].set_xlabel("Value")
    axes[0].set_ylabel("Count")
    axes[0].legend()
    
    # B: ROOT Direct
    axes[1].hist(root_result, bins=50, alpha=0.7, color='orange', label='ROOT')
    axes[1].set_title(f"B: ROOT Direct\n{name}")
    axes[1].set_xlabel("Value")
    axes[1].legend()
    
    # A-B: Difference
    diff = np.array(dsl_result) - np.array(root_result)
    max_diff = np.max(np.abs(diff))
    
    axes[2].hist(diff, bins=50, color='red', alpha=0.7)
    axes[2].axvline(0, color='k', linestyle='--', linewidth=2)
    axes[2].set_title(f"A-B: Difference\nmax|diff| = {max_diff:.2e}")
    axes[2].set_xlabel("DSL - ROOT")
    
    plt.tight_layout()
    plt.show()
    
    # Assertion
    passed = np.allclose(dsl_result, root_result, atol=tolerance)
    if passed:
        print(f"✓ {name}: Invariance verified (max|diff| = {max_diff:.2e})")
    else:
        print(f"✗ {name}: FAILED (max|diff| = {max_diff:.2e})")
    
    return passed

## Generate Test Data

In [ ]:
from tests.generators.toy_nd import generate_nd_2d_root

filename = generate_nd_2d_root(size='M', seed=42)  # Medium size for better statistics
rdf = ROOT.RDataFrame("Events", filename)

print(f"Events: {rdf.Count().GetValue()}")

schema = {
    'event_id': 'long',
    'n_tracks': 'int',
    'event_weight': 'double',
    'track_pt': 'RVec<double>',
    'track_eta': 'RVec<double>',
    'cluster_Q': 'RVec<RVec<double>>',
}

## Invariance Check #1: Scalar Arithmetic

**Operation:** `event_weight * 2`

In [ ]:
# DSL
dsl1 = DSLCompiler(schema)
dsl1.define("scaled", "event_weight * 2")
df_dsl = dsl1.to_pandas(rdf, ['scaled'])

# ROOT Direct
data = rdf.AsNumpy(['event_weight'])
root_result = data['event_weight'] * 2

# Compare
check1 = visual_invariance_check("event_weight * 2", df_dsl['scaled'].values, root_result)

## Invariance Check #2: Sum Reduction

**Operation:** `Sum(track_pt)`

In [ ]:
# DSL
dsl2 = DSLCompiler(schema)
dsl2.define("sum_pt", "Sum(track_pt)")
df_dsl2 = dsl2.to_pandas(rdf, ['sum_pt'])

# Python computation - convert RVec to numpy
data2 = rdf.AsNumpy(['track_pt'])
root_result2 = np.array([np.sum(np.array(pt)) for pt in data2['track_pt']])

# Compare
check2 = visual_invariance_check("Sum(track_pt)", df_dsl2['sum_pt'].values, root_result2)

## Invariance Check #3: Mean Reduction

**Operation:** `Mean(track_pt)`

In [ ]:
# DSL
dsl3 = DSLCompiler(schema)
dsl3.define("mean_pt", "Mean(track_pt)")
df_dsl3 = dsl3.to_pandas(rdf, ['mean_pt'])

# Python computation - convert RVec to numpy
root_result3 = np.array([np.mean(np.array(pt)) if len(pt) > 0 else 0 for pt in data2['track_pt']])

# Compare
check3 = visual_invariance_check("Mean(track_pt)", df_dsl3['mean_pt'].values, root_result3)

## Invariance Check #4: Max Reduction

**Operation:** `Max(track_pt)`

In [ ]:
# DSL
dsl4 = DSLCompiler(schema)
dsl4.define("max_pt", "Max(track_pt)")
df_dsl4 = dsl4.to_pandas(rdf, ['max_pt'])

# Python computation - convert RVec to numpy
root_result4 = np.array([np.max(np.array(pt)) if len(pt) > 0 else 0 for pt in data2['track_pt']])

# Compare
check4 = visual_invariance_check("Max(track_pt)", df_dsl4['max_pt'].values, root_result4)

## Invariance Check #5: 1D Transform

**Operation:** `track_pt / 1000`

Flattened comparison (all track pT values).

In [ ]:
# DSL
dsl5 = DSLCompiler(schema)
dsl5.define("pt_scaled", "track_pt / 1000")
df_dsl5 = dsl5.to_pandas(rdf, ['pt_scaled'])

# Python computation - flatten - convert RVec to numpy
root_flat = np.concatenate([np.array(pt) / 1000 for pt in data2['track_pt']])

# Compare (flattened)
check5 = visual_invariance_check("track_pt / 1000 (flattened)", 
                                  df_dsl5['pt_scaled'].values, root_flat)

## Invariance Check #6: sqrt Function

**Operation:** `sqrt(track_pt)`

In [ ]:
# DSL
dsl6 = DSLCompiler(schema)
dsl6.define("sqrt_pt", "sqrt(track_pt)")
df_dsl6 = dsl6.to_pandas(rdf, ['sqrt_pt'])

# Python computation - flatten - convert RVec to numpy
root_flat6 = np.concatenate([np.sqrt(np.array(pt)) for pt in data2['track_pt']])

# Compare
check6 = visual_invariance_check("sqrt(track_pt) (flattened)", 
                                  df_dsl6['sqrt_pt'].values, root_flat6)

## Invariance Check #7: 2D Nested Sum

**Operation:** `Sum(cluster_Q)`

In [ ]:
# DSL
dsl7 = DSLCompiler(schema)
dsl7.define("total_Q", "Sum(cluster_Q)")
df_dsl7 = dsl7.to_pandas(rdf, ['total_Q'])

# Python computation - convert RVec to numpy
data7 = rdf.AsNumpy(['cluster_Q'])
root_result7 = np.array([sum(sum(np.array(t)) for t in evt) for evt in data7['cluster_Q']])

# Compare
check7 = visual_invariance_check("Sum(cluster_Q)", df_dsl7['total_Q'].values, root_result7)

## Invariance Check #8: Element-wise Scaling

**Operation:** `track_pt * 2`

In [ ]:
# DSL
dsl8 = DSLCompiler(schema)
dsl8.define("scaled_pt", "track_pt * 2")
df_dsl8 = dsl8.to_pandas(rdf, ['scaled_pt'])

# Python computation - convert RVec to numpy
data8 = rdf.AsNumpy(['track_pt'])
root_flat8 = np.concatenate([np.array(pt) * 2 for pt in data8['track_pt']])

# Compare
check8 = visual_invariance_check("track_pt * 2 (element-wise)", 
                                  df_dsl8['scaled_pt'].values, root_flat8)

## Invariance Check #9: Combined Operation

**Operation:** `sqrt(track_pt**2 + track_eta**2)`

In [ ]:
# DSL
dsl9 = DSLCompiler(schema)
dsl9.define("momentum", "sqrt(track_pt**2 + track_eta**2)")
df_dsl9 = dsl9.to_pandas(rdf, ['momentum'])

# Python computation - convert RVec to numpy
data9 = rdf.AsNumpy(['track_pt', 'track_eta'])
root_flat9 = []
for pt, eta in zip(data9['track_pt'], data9['track_eta']):
    pt_np = np.array(pt)  # Convert RVec to numpy
    eta_np = np.array(eta)
    root_flat9.extend(np.sqrt(pt_np**2 + eta_np**2))
root_flat9 = np.array(root_flat9)

# Compare
check9 = visual_invariance_check("sqrt(pt² + eta²) (flattened)", 
                                  df_dsl9['momentum'].values, root_flat9)

## Invariance Check #10: Comparison Operation

**Operation:** `track_pt > 5`

In [ ]:
# DSL
dsl10 = DSLCompiler(schema)
dsl10.define("high_pt", "track_pt > 5")
df_dsl10 = dsl10.to_pandas(rdf, ['high_pt'])

# Python computation - convert RVec to numpy
root_flat10 = np.concatenate([np.array(pt) > 5 for pt in data2['track_pt']])

# Compare (as int for histogram)
check10 = visual_invariance_check("track_pt > 5 (flattened)", 
                                   df_dsl10['high_pt'].values.astype(int), 
                                   root_flat10.astype(int))

## Summary

In [ ]:
print("=" * 60)
print("08f_invariance_visual.ipynb - A/B/A-B Comparison")
print("=" * 60)

checks = [
    ("#1 Scalar arithmetic (event_weight * 2)", check1),
    ("#2 Sum reduction (Sum(track_pt))", check2),
    ("#3 Mean reduction (Mean(track_pt))", check3),
    ("#4 Max reduction (Max(track_pt))", check4),
    ("#5 1D transform (track_pt / 1000)", check5),
    ("#6 sqrt function (sqrt(track_pt))", check6),
    ("#7 2D nested sum (Sum(cluster_Q))", check7),
    ("#8 Element-wise scaling", check8),
    ("#9 Combined operation", check9),
    ("#10 Comparison operation", check10),
]

passed = 0
for name, result in checks:
    status = "✓ PASSED" if result else "✗ FAILED"
    print(f"{name}: {status}")
    if result:
        passed += 1

print("\n" + "=" * 60)
print(f"TOTAL: {passed}/{len(checks)} invariance checks passed")
print("=" * 60)

if passed == len(checks):
    print("\n🎉 ALL INVARIANCE CHECKS PASSED 🎉")
else:
    print(f"\n⚠️ {len(checks) - passed} checks failed")